# Precip_ocean_land

In [1]:
import sys
sys.path.append('/work/mh0731/m300876/package/')
import icons
import intake
import matplotlib.pyplot as plt
import healpy as hp
import xarray as xr
import numpy as np
import easygems.healpix as egh
import seaborn as sns
import cmocean

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
icons.prepare_cpu(memory='100GB')

Number of CPUs: 256, number of threads: 256, number of workers: 1, processes: False


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/m300876/levante-spawner-preset//proxy/43961/status,
Dashboard: /user/m300876/levante-spawner-preset//proxy/43961/status,Workers: 1
Total threads: 256,Total memory: 93.13 GiB
Status: running,Using processes: False
Comm: inproc://136.172.122.122/2433575/1,Workers: 1
Dashboard: /user/m300876/levante-spawner-preset//proxy/43961/status,Total threads: 256
Started: Just now,Total memory: 93.13 GiB
Comm: inproc://136.172.122.122/2433575/4,Total threads: 256
Dashboard: /user/m300876/levante-spawner-preset//proxy/42761/status,Memory: 93.13 GiB
Nanny: None,


### Predefined functions

In [6]:
### Location mask
def tropics(ds):
    return np.abs(ds.lat) <= 30.1

def ocean(ds):
    return (np.isnan(ds.sftlf))

def land(ds):
    return (ds.sftlf == 1)

## Calling data

In [4]:
time_slice = slice("2020-02-01", "2021-01-31")

In [5]:
%%time
### ICON
current_location = "online"
cat = intake.open_catalog(
    "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")[current_location]
exp_name = "icon_d3hp003"
experiment = cat[exp_name]

ds_icon = (
    experiment(zoom=8,time="P1D", chunks="auto")
    .to_dask()
    .pipe(egh.attach_coords)
    .sel(time=time_slice)
)

CPU times: user 1.66 s, sys: 653 ms, total: 2.31 s
Wall time: 6.43 s


In [7]:
pr_icon_land = ds_icon.where(tropics(ds_icon) & land(ds_icon)).mean('cell').compute()
pr_icon_ocean = ds_icon.where(tropics(ds_icon) & ocean(ds_icon)).mean('cell').compute()